# Stage 1 — CREsted topic model evaluation FULL v2

This notebook is intentionally self-contained: config, helper functions, and execution are all included.

In [ ]:
from __future__ import annotations

import os
os.environ.setdefault("KERAS_BACKEND", "torch")

from pathlib import Path
import warnings

import anndata as ad
import crested
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics import average_precision_score, roc_auc_score, accuracy_score, confusion_matrix

# =====================
# Config
# =====================
BASE = Path("/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt")
DATA = BASE / "data"
OUTDIR = BASE / "runs" / "out" / "stage1_crested_eval_v2"
OUTDIR.mkdir(parents=True, exist_ok=True)

RUN_SPECIES = ["human", "macaque"]  # change to ["human"] first if you want to test
MODEL_LAYER = "model_prediction"

DATASETS = {
    "human": {
        "adata": BASE / "runs" / "out" / "human_topics.h5ad",
        "model": BASE / "runs" / "out" / "deeptopic_human" / "final_model.keras",
        "genome_fasta": BASE.parent / "genomes" / "hg38" / "hg38.fa",
        "chrom_sizes": BASE.parent / "genomes" / "hg38" / "hg38.chrom.sizes",
        "annotation_candidates": [
            DATA / "3_topic_annotation_human_pycistopic.tsv",
            DATA / "topic_annotation_human_pycistopic.tsv",
            DATA / "human_topic_annotation_pycistopic.tsv",
        ],
        "qc_candidates": [
            DATA / "3_topic_qc_metrics_human.tsv",
            DATA / "topic_qc_metrics_human.tsv",
            DATA / "human_topic_qc_metrics.tsv",
        ],
    },
    "macaque": {
        "adata": BASE / "runs" / "out" / "macaque_topics.h5ad",
        "model": BASE / "runs" / "out" / "deeptopic_macaque" / "final_model.keras",
        "genome_fasta": BASE.parent / "yiquan" / "macaque" / "rheMac10.fa",
        "chrom_sizes": None,
        "annotation_candidates": [
            DATA / "3_topic_annotation_macaque_pycistopic.tsv",
            DATA / "topic_annotation_macaque_pycistopic.tsv",
            DATA / "macaque_topic_annotation_pycistopic.tsv",
        ],
        "qc_candidates": [
            DATA / "3_topic_qc_metrics_macaque.tsv",
            DATA / "topic_qc_metrics_macaque.tsv",
            DATA / "macaque_topic_qc_metrics.tsv",
        ],
    },
}

# =====================
# Helpers
# =====================
def first_existing(candidates):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    return None


def read_table_auto(path: Path | None):
    if path is None:
        return None
    sep = "\t" if str(path).endswith((".tsv", ".txt")) else ","
    return pd.read_csv(path, sep=sep)


def normalize_topic_name(x):
    s = str(x)
    if s.startswith("Topic"):
        return s
    try:
        return f"Topic{int(float(s))}"
    except Exception:
        return s


def load_topic_annotation(candidates):
    path = first_existing(candidates)
    if path is None:
        print("[WARN] no topic annotation table found")
        return None
    print("[load annotation]", path)
    df = read_table_auto(path)
    # pycisTopic sometimes writes unnamed index column
    if "topic" not in df.columns:
        first = df.columns[0]
        df = df.rename(columns={first: "topic"})
    df["topic"] = df["topic"].map(normalize_topic_name)
    return df


def load_topic_qc(candidates):
    path = first_existing(candidates)
    if path is None:
        print("[WARN] no topic QC table found")
        return None
    print("[load QC]", path)
    df = read_table_auto(path)
    if "topic" not in df.columns:
        first = df.columns[0]
        df = df.rename(columns={first: "topic"})
    df["topic"] = df["topic"].map(normalize_topic_name)
    return df


def attach_annotation_to_obs(adata, annot_df=None, qc_df=None):
    # CREsted topic h5ad layout: obs = classes/topics, var = regions.
    adata.obs["topic"] = [normalize_topic_name(x) for x in adata.obs_names]
    if annot_df is not None:
        add = annot_df.drop_duplicates("topic").set_index("topic")
        for col in add.columns:
            adata.obs[col] = adata.obs["topic"].map(add[col])
    if qc_df is not None:
        add = qc_df.drop_duplicates("topic").set_index("topic")
        for col in add.columns:
            new_col = col if col not in adata.obs.columns else f"{col}_qc"
            adata.obs[new_col] = adata.obs["topic"].map(add[col])
    print("[obs annotation columns]", list(adata.obs.columns))


def register_species_genome(cfg):
    fasta = cfg["genome_fasta"]
    chrom_sizes = cfg.get("chrom_sizes")
    if chrom_sizes is not None and Path(chrom_sizes).exists():
        genome = crested.Genome(str(fasta), str(chrom_sizes))
    else:
        genome = crested.Genome(str(fasta))
    crested.register_genome(genome)
    return genome


def ensure_split(adata):
    if "split" in adata.var.columns:
        print("[split exists]")
        print(adata.var["split"].value_counts(dropna=False))
        return
    print("[WARN] no split column found; using CREsted chromosome split fallback")
    crested.pp.train_val_test_split(
        adata,
        strategy="chr",
        val_chroms=["chr8", "chr10"],
        test_chroms=["chr9", "chr18"],
    )
    print(adata.var["split"].value_counts(dropna=False))


def add_predictions_to_layer(adata, model, layer_name):
    print("[predict] crested.tl.predict")
    pred = crested.tl.predict(adata, model=model)
    print("[predict shape]", pred.shape)
    # CREsted tutorial stores predictions.T because AnnData layers are classes x regions.
    adata.layers[layer_name] = pred.T
    print(f"[layer saved] adata.layers['{layer_name}'] shape =", adata.layers[layer_name].shape)


def dense_array(x):
    return x.toarray() if sparse.issparse(x) else np.asarray(x)


def get_split_mask(adata, split="test"):
    if "split" not in adata.var.columns:
        return np.ones(adata.n_vars, dtype=bool)
    return adata.var["split"].astype(str).values == split


def compute_per_topic_metrics(adata, split="test", layer_name=MODEL_LAYER):
    mask = get_split_mask(adata, split)
    y_true = dense_array(adata.X)[:, mask]
    y_pred = dense_array(adata.layers[layer_name])[:, mask]
    rows = []
    for i, topic in enumerate(adata.obs_names):
        yt = y_true[i]
        yp = y_pred[i]
        # binarized topic labels are expected, but keep it robust
        yt_bin = (yt > 0).astype(int)
        try:
            aupr = average_precision_score(yt_bin, yp)
        except Exception:
            aupr = np.nan
        try:
            auroc = roc_auc_score(yt_bin, yp) if len(np.unique(yt_bin)) > 1 else np.nan
        except Exception:
            auroc = np.nan
        rows.append({
            "topic": normalize_topic_name(topic),
            "n_positive_regions": int(yt_bin.sum()),
            "mean_true": float(np.mean(yt)),
            "mean_pred": float(np.mean(yp)),
            "auPR": aupr,
            "auROC": auroc,
        })
    metrics = pd.DataFrame(rows)
    true_argmax = y_true.argmax(axis=0)
    pred_argmax = y_pred.argmax(axis=0)
    overall = pd.DataFrame([{
        "split": split,
        "n_regions": int(mask.sum()),
        "argmax_accuracy": accuracy_score(true_argmax, pred_argmax),
        "mean_auPR": metrics["auPR"].mean(),
        "mean_auROC": metrics["auROC"].mean(),
    }])
    return metrics, overall, true_argmax, pred_argmax


def make_confusion_table(adata, true_argmax, pred_argmax):
    labels = [normalize_topic_name(x) for x in adata.obs_names]
    cm = confusion_matrix(true_argmax, pred_argmax, labels=list(range(len(labels))))
    return pd.DataFrame(cm, index=labels, columns=labels)


def plot_per_topic_bars(metrics, out, col):
    df = metrics.copy().sort_values(col, ascending=False)
    plt.figure(figsize=(18, 5))
    plt.bar(df["topic"], df[col])
    plt.xticks(rotation=90, fontsize=6)
    plt.ylabel(col)
    plt.tight_layout()
    plt.savefig(out, dpi=220, bbox_inches="tight")
    plt.close()
    print("[save fig]", out)


def plot_confusion(cm_df, out):
    if cm_df is None:
        return
    plt.figure(figsize=(16, 14))
    plt.imshow(cm_df.values, aspect="auto")
    plt.xticks(np.arange(cm_df.shape[1]), cm_df.columns, rotation=90, fontsize=5)
    plt.yticks(np.arange(cm_df.shape[0]), cm_df.index, fontsize=5)
    plt.colorbar(fraction=0.02, pad=0.01, label="count")
    plt.title("Argmax topic confusion on test regions")
    plt.tight_layout()
    plt.savefig(out, dpi=220, bbox_inches="tight")
    plt.close()
    print("[save fig]", out)


def run_one_species(tag, cfg):
    print(f"\n===== {tag} =====")
    species_out = OUTDIR / tag
    species_out.mkdir(parents=True, exist_ok=True)

    print("[load adata]", cfg["adata"])
    adata = ad.read_h5ad(cfg["adata"])
    print(adata)

    annot_df = load_topic_annotation(cfg["annotation_candidates"])
    qc_df = load_topic_qc(cfg["qc_candidates"])
    attach_annotation_to_obs(adata, annot_df, qc_df)
    adata.obs.to_csv(species_out / f"{tag}_class_topic_annotation_used.tsv", sep="\t")

    print("[register genome]")
    register_species_genome(cfg)
    ensure_split(adata)

    print("[load model]", cfg["model"])
    model = crested.utils.load_model(str(cfg["model"]))
    print(model)

    # Tutorial style: predict then store predictions.T in an AnnData layer.
    add_predictions_to_layer(adata, model, MODEL_LAYER)

    # Tutorial style: evaluate on test set.
    print("[crested evaluate] test set")
    try:
        eval_result = crested.tl.evaluate(
            adata,
            model=MODEL_LAYER,
            metrics=crested.tl.default_configs("topic_classification"),
        )
        print(eval_result)
        pd.DataFrame(eval_result if isinstance(eval_result, list) else [eval_result]).to_csv(
            species_out / f"{tag}_crested_evaluate.tsv", sep="\t", index=False
        )
    except Exception as e:
        print("[WARN] crested.tl.evaluate failed; custom sklearn metrics will still be exported.")
        print(type(e).__name__, e)

    metrics, overall, true_argmax, pred_argmax = compute_per_topic_metrics(adata, split="test", layer_name=MODEL_LAYER)
    if annot_df is not None:
        metrics = metrics.merge(annot_df, on="topic", how="left")
    if qc_df is not None:
        metrics = metrics.merge(qc_df, on="topic", how="left", suffixes=("", "_qc"))

    metrics.to_csv(species_out / f"{tag}_per_topic_test_metrics.tsv", sep="\t", index=False)
    overall.to_csv(species_out / f"{tag}_overall_test_metrics.tsv", sep="\t", index=False)
    print("[save metrics]", species_out)

    cm_df = make_confusion_table(adata, true_argmax, pred_argmax)
    cm_df.to_csv(species_out / f"{tag}_argmax_confusion_counts.tsv", sep="\t")

    plot_per_topic_bars(metrics, species_out / f"{tag}_per_topic_auPR.png", "auPR")
    plot_per_topic_bars(metrics, species_out / f"{tag}_per_topic_auROC.png", "auROC")
    plot_confusion(cm_df, species_out / f"{tag}_argmax_confusion_heatmap.png")

    try:
        top_topic = metrics.sort_values("auPR", ascending=False)["topic"].iloc[0]
        print("[plot scatter] top topic:", top_topic)
        crested.pl.corr.scatter(
            adata,
            class_name=top_topic,
            model_names=MODEL_LAYER,
            split="test",
            log_transform=False,
            square=True,
            width=8,
            height=8,
        )
        plt.savefig(species_out / f"{tag}_scatter_{top_topic}.png", dpi=220, bbox_inches="tight")
        plt.close()
    except Exception as e:
        print("[WARN] crested.pl.corr.scatter failed:", type(e).__name__, e)

    try:
        crested.pl.corr.heatmap(adata, split="test", log_transform=False, vmax=1, vmin=0)
        plt.savefig(species_out / f"{tag}_crested_corr_heatmap.png", dpi=220, bbox_inches="tight")
        plt.close()
    except Exception as e:
        print("[WARN] crested.pl.corr.heatmap failed:", type(e).__name__, e)

    out_h5ad = species_out / f"{tag}_topics_with_predictions_stage1.h5ad"
    adata.write_h5ad(out_h5ad)
    print("[save]", out_h5ad)
    return adata, metrics, overall


results = {}
for tag in RUN_SPECIES:
    results[tag] = run_one_species(tag, DATASETS[tag])
